# sol03: Reliability-Focused Incident Triage Agent

Contains:
- the same scenario as `03_mock`
- one complete reference implementation
- grading tests


In [ ]:
import inspect
import json
from copy import deepcopy
from typing import Any, Callable

LOOKUP_ATTEMPTS: dict[str, int] = {}


def reset_state() -> None:
    LOOKUP_ATTEMPTS.clear()


def fetch_ticket(ticket_id: str) -> dict[str, Any]:
    return {
        "ticket_id": ticket_id,
        "service": "payments" if ticket_id == "inc-1" else "billing",
        "severity": "high",
    }


def lookup_runbook(service: str) -> dict[str, Any]:
    count = LOOKUP_ATTEMPTS.get(service, 0)
    LOOKUP_ATTEMPTS[service] = count + 1
    if service == "payments" and count == 0:
        raise RuntimeError("transient backend timeout")
    return {"service": service, "playbook": f"restart_{service}_workers"}


TOOL_REGISTRY: dict[str, Callable[..., Any]] = {
    "fetch_ticket": fetch_ticket,
    "lookup_runbook": lookup_runbook,
}


class TriageModel:
    def __init__(self, scenario: str) -> None:
        self.scenario = scenario
        self.step = 0

    def __call__(self, messages: list[dict[str, Any]]) -> dict[str, Any]:
        self.step += 1

        if self.scenario == "cache":
            if self.step == 1:
                return {"stop_reason": "tool_use", "tool_calls": [{"id": "a1", "name": "lookup_runbook", "input": {"service": "billing"}}]}
            if self.step == 2:
                return {"stop_reason": "tool_use", "tool_calls": [{"id": "a2", "name": "lookup_runbook", "input": {"service": "billing"}}]}
            return {
                "stop_reason": "end_turn",
                "output_text": json.dumps({"summary": "billing issue mitigated", "action": "restart_billing_workers", "confidence": 0.78}),
            }

        if self.scenario == "retry":
            if self.step == 1:
                return {"stop_reason": "tool_use", "tool_calls": [{"id": "b1", "name": "lookup_runbook", "input": {"service": "payments"}}]}
            return {
                "stop_reason": "end_turn",
                "output_text": json.dumps({"summary": "payments issue mitigated", "action": "restart_payments_workers", "confidence": 0.81}),
            }

        if self.scenario == "loop":
            return {"stop_reason": "tool_use", "tool_calls": [{"id": "loop", "name": "fetch_ticket", "input": {"ticket_id": "inc-1"}}]}

        return {"stop_reason": "end_turn", "output_text": "{}"}


In [ ]:
def _validate_tool_call(tool_call: dict[str, Any], tool_registry: dict[str, Callable[..., Any]]) -> str | None:
    for key in ("id", "name", "input"):
        if key not in tool_call:
            return "tool_call_missing_required_fields"

    name = tool_call["name"]
    payload = tool_call["input"]
    if name not in tool_registry:
        return "unknown_tool"
    if not isinstance(payload, dict):
        return "tool_input_must_be_object"

    sig = inspect.signature(tool_registry[name])
    missing = [
        param.name
        for param in sig.parameters.values()
        if param.default is inspect._empty and param.name not in payload
    ]
    if missing:
        return f"missing_required_args:{','.join(sorted(missing))}"
    return None


def execute_tool_call(
    tool_call: dict[str, Any],
    tool_registry: dict[str, Callable[..., Any]],
    cache: dict[str, dict[str, Any]],
) -> dict[str, Any]:
    tool_id = str(tool_call.get("id", "missing_id"))
    tool_name = str(tool_call.get("name", "missing_name"))
    payload = tool_call.get("input", {})
    cache_key = f"{tool_name}:{json.dumps(payload, sort_keys=True)}"

    err = _validate_tool_call(tool_call, tool_registry)
    if err:
        return {
            "role": "tool",
            "tool_call_id": tool_id,
            "name": tool_name,
            "is_error": True,
            "from_cache": False,
            "content": json.dumps({"error": err}, sort_keys=True),
        }

    if cache_key in cache:
        cached = deepcopy(cache[cache_key])
        cached["tool_call_id"] = tool_id
        cached["from_cache"] = True
        return cached

    attempt = 0
    while True:
        attempt += 1
        try:
            result = tool_registry[tool_name](**payload)
            msg = {
                "role": "tool",
                "tool_call_id": tool_id,
                "name": tool_name,
                "is_error": False,
                "from_cache": False,
                "content": json.dumps({"result": result}, sort_keys=True),
            }
            cache[cache_key] = deepcopy(msg)
            return msg
        except RuntimeError as exc:
            transient = "transient" in str(exc).lower()
            if transient and attempt == 1:
                continue
            return {
                "role": "tool",
                "tool_call_id": tool_id,
                "name": tool_name,
                "is_error": True,
                "from_cache": False,
                "content": json.dumps({"error": str(exc)}, sort_keys=True),
            }
        except Exception as exc:  # pragma: no cover
            return {
                "role": "tool",
                "tool_call_id": tool_id,
                "name": tool_name,
                "is_error": True,
                "from_cache": False,
                "content": json.dumps({"error": str(exc)}, sort_keys=True),
            }


def parse_final_output(output_text: str) -> dict[str, Any]:
    data = json.loads(output_text)
    required = {"summary", "action", "confidence"}
    missing = required.difference(data)
    if missing:
        raise ValueError(f"missing_final_keys:{','.join(sorted(missing))}")
    if not isinstance(data["confidence"], (int, float)):
        raise ValueError("confidence_must_be_numeric")
    return data


def run_agent(
    user_prompt: str,
    model: Callable[[list[dict[str, Any]]], dict[str, Any]],
    tool_registry: dict[str, Callable[..., Any]],
    max_steps: int = 6,
) -> dict[str, Any]:
    messages: list[dict[str, Any]] = [{"role": "user", "content": user_prompt}]
    cache: dict[str, dict[str, Any]] = {}
    stats = {"tool_calls": 0, "cache_hits": 0}

    for _ in range(max_steps):
        response = model(messages)
        stop_reason = response.get("stop_reason")

        if stop_reason == "tool_use":
            tool_calls = response.get("tool_calls", [])
            if not isinstance(tool_calls, list):
                raise RuntimeError("tool_calls_must_be_list")
            for tool_call in tool_calls:
                tool_msg = execute_tool_call(tool_call, tool_registry, cache)
                stats["tool_calls"] += 1
                if tool_msg.get("from_cache"):
                    stats["cache_hits"] += 1
                messages.append(tool_msg)
            continue

        if stop_reason == "end_turn":
            final = parse_final_output(str(response.get("output_text", "{}")))
            return {"final": final, "messages": messages, "stats": stats}

        raise RuntimeError(f"unsupported_stop_reason:{stop_reason}")

    raise RuntimeError("max_steps_exceeded")


In [ ]:
def run_exam03_tests() -> None:
    reset_state()

    # 1) Cache behavior
    model = TriageModel("cache")
    result = run_agent("triage billing", model, TOOL_REGISTRY)
    assert result["stats"]["cache_hits"] == 1
    assert LOOKUP_ATTEMPTS["billing"] == 1

    # 2) Retry behavior for transient errors
    reset_state()
    model = TriageModel("retry")
    result = run_agent("triage payments", model, TOOL_REGISTRY)
    assert LOOKUP_ATTEMPTS["payments"] == 2
    assert result["final"]["action"] == "restart_payments_workers"

    # 3) Structured final output required
    assert {"summary", "action", "confidence"}.issubset(result["final"].keys())

    # 4) Max step protection
    model = TriageModel("loop")
    try:
        run_agent("loop", model, TOOL_REGISTRY, max_steps=3)
        raise AssertionError("Expected max_steps_exceeded")
    except RuntimeError as exc:
        assert "max_steps_exceeded" in str(exc)

    print("03_mock tests passed")


run_exam03_tests()


## Walkthrough: Exactly How to Solve `03_mock`

### 0) First 3 minutes
- This is reliability-first, not algorithm-first.
- Write these goals in comments: cache hit, one retry, strict final JSON parse.

### 1) Should I read tests now?
Yes, because tests define reliability policy:
- `cache_hits == 1`
- `LOOKUP_ATTEMPTS["payments"] == 2` (one retry happened)
- final output must include `summary/action/confidence`
- max step protection must raise error

### 2) Coding order
1. `parse_final_output` (small, deterministic).
2. `execute_tool_call` with:
   - validation
   - cache read/write
   - one retry around transient tool failure
3. `run_agent`:
   - maintain `stats`
   - increment `tool_calls`
   - increment `cache_hits` from tool message metadata

### 3) One concrete example to narrate aloud
Use retry scenario:
- First `lookup_runbook(payments)` throws transient timeout.
- Retry once, second attempt succeeds.
- Final action becomes `restart_payments_workers`.
- You can explain this as "bounded retry for transient faults."

### 4) What to say while coding
- "I’m defining explicit reliability policy first, then implementing to that contract."
- "Cache key is tool-name plus normalized input to reduce duplicate work."
- "Retry is bounded to one attempt to avoid runaway latency."
- "I validate final output schema to prevent silent bad responses."

### 5) Self-check before final run
- Can cached calls skip tool execution?
- Is retry only for runtime failures, not validation failures?
- Do stats reflect actual behavior?
